# 第三部分 3.3：循环神经网络（RNN）

| 章节 | 内容 |
|------|------|
| **3.5 循环神经网络** | 序列建模、梯度消失、LSTM / GRU |

---

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('环境就绪，PyTorch 版本：', torch.__version__)
print('使用设备：', device)

## 3.5 循环神经网络（RNN）

### 序列数据的特殊性

CNN 和 MLP 的输入假设：**样本之间相互独立**，每张图像与其他图像无关。

但很多真实数据是**序列**——当前的值依赖于之前的历史：
- 文本：「我喜欢__」，下一个词依赖前面的句子结构
- 时间序列：今天的股价依赖过去几天的走势
- 语音：当前的音频帧依赖前面的发音
- 视频：当前帧依赖前序帧

CNN / MLP 处理序列的困境：
- **输入长度固定**：这些模型的输入维度在设计时就固定了，无法处理任意长度的序列
- **无顺序感知**：MLP 把序列展平，丢失了「谁在谁前面」的信息
- **不共享时间权重**：第 1 步和第 5 步的模式识别应该用同一套参数，MLP 对每个位置用不同权重

---

### RNN 的基本结构

RNN 的核心想法：维护一个**隐藏状态（Hidden State）** $h_t$，作为「记忆」，在每个时间步更新：

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

$$y_t = W_y h_t + b_y$$

- $x_t$：当前时间步的输入
- $h_{t-1}$：上一时间步的隐藏状态（记忆）
- $h_t$：当前时间步的新隐藏状态
- $y_t$：当前时间步的输出（可选，看任务需要）

**关键**：$W_h, W_x, b$ 在所有时间步共享——无论序列多长，参数数量不变。

```
x₁ → [RNN Cell] → h₁ → [RNN Cell] → h₂ → [RNN Cell] → h₃ → 输出
           ↑                  ↑                  ↑
         h₀=0              h₁               h₂

同一套参数 W_h, W_x 在每个时间步复用
```

---

### RNN 的任务类型

| 类型 | 输入 | 输出 | 典型应用 |
|------|------|------|----------|
| **多对一（Many-to-One）** | 序列 | 单个值 | 情感分类（一段文字 → 正/负面）、序列分类 |
| **一对多（One-to-Many）** | 单个值 | 序列 | 图像描述生成（一张图 → 一段话）|
| **多对多（Many-to-Many）** | 序列 | 序列 | 机器翻译（中文序列 → 英文序列）、命名实体识别 |

---

### RNN 的梯度消失问题

RNN 的训练通过**时序反向传播（BPTT, Backpropagation Through Time）** 完成——将 RNN 在时间上「展开」成一个深层网络，然后用普通反向传播计算梯度。

**问题**：展开后的网络可能有几百层（长序列有多少步就展开多少层）。梯度从最后一步反传到最初一步，需要经过所有时间步的矩阵乘法。如果 $\|W_h\| < 1$，梯度指数级消失；如果 $\|W_h\| > 1$，梯度指数级爆炸。

实际影响：**RNN 很难学到长距离依赖**。「它早年学过钢琴，所以他后来的__」——「他」依赖序列开头的「它」，但这种跨越很多步的依赖，普通 RNN 几乎无法学到。

---

### LSTM：门控记忆

LSTM（Long Short-Term Memory，1997）通过引入**门控机制（Gating）** 和一条独立的**细胞状态（Cell State）** 来解决梯度消失问题。

细胞状态 $C_t$ 是 LSTM 的核心——它像一条「传送带」，信息可以直接从头传到尾，几乎不经过非线性变换，梯度可以几乎无损地反向传播。

**三个门控制细胞状态的读写：**

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \qquad \text{（遗忘门：该忘掉多少旧记忆）}$$

$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i), \quad \tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C) \qquad \text{（输入门：该写入多少新信息）}$$

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \qquad \text{（更新细胞状态：遗忘旧的 + 写入新的）}$$

$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o), \quad h_t = o_t \odot \tanh(C_t) \qquad \text{（输出门：该输出多少）}$$

其中 $\odot$ 是逐元素乘法（Hadamard 积），$\sigma$ 是 Sigmoid 函数（输出 0~1，控制门开/关程度）。

**直觉类比**：
- **遗忘门**：「我读到第 10 句，发现主语换了，之前记住的性别信息可以忘掉了」
- **输入门**：「这一步出现了重要词汇，我要把它写入长期记忆」
- **输出门**：「当前任务需要读取细胞状态里的哪部分来形成隐藏状态」

---

### GRU：简化版 LSTM

GRU（Gated Recurrent Unit，2014）把 LSTM 的三个门简化为两个，并合并细胞状态和隐藏状态：

$$z_t = \sigma(W_z [h_{t-1}, x_t]) \qquad \text{（更新门，控制新旧信息比例）}$$

$$r_t = \sigma(W_r [h_{t-1}, x_t]) \qquad \text{（重置门，控制旧记忆的读取程度）}$$

$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t]) \qquad \text{（候选新状态）}$$

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \qquad \text{（线性插值：保留旧的 + 写入新的）}$$

| | RNN | LSTM | GRU |
|-|-----|------|-----|
| 状态 | $h_t$ | $h_t$ + $C_t$ | $h_t$ |
| 门数量 | 0 | 3 个门 | 2 个门 |
| 参数量 | 最少 | 最多（约 4× RNN）| 中等（约 3× RNN）|
| 长程依赖 | 差 | 好 | 好（略逊于 LSTM）|
| 训练速度 | 快 | 慢 | 中等 |

> **实践建议**：
> - 优先尝试 **GRU**（参数更少，训练更快，性能接近 LSTM）
> - 需要极高精度或超长序列时用 **LSTM**
> - 在 NLP 中，大多数任务已被 **Transformer** 取代，LSTM/GRU 主要用于时间序列、嵌入式设备等对推理速度有严格要求的场景

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ── 演示 1：梯度消失 vs LSTM/GRU ──────────────────────────────
# 用一个简单实验说明普通 RNN 在长序列上梯度消失有多严重

def compute_first_step_grad(rnn_type, seq_len, hidden_size=32, batch=8, input_size=16):
    """
    构建一个 rnn_type 单元，输入长度为 seq_len 的序列，
    计算损失后做反向传播，返回第一个时间步输入处的梯度范数。
    梯度范数越小 → 梯度消失越严重 → 模型越难学到长程依赖。
    """
    torch.manual_seed(0)

    if rnn_type == 'RNN':
        cell = nn.RNN(input_size, hidden_size, batch_first=True)
    elif rnn_type == 'LSTM':
        cell = nn.LSTM(input_size, hidden_size, batch_first=True)
    else:
        cell = nn.GRU(input_size, hidden_size, batch_first=True)

    x = torch.randn(batch, seq_len, input_size, requires_grad=True)

    out, _ = cell(x)               # out: (batch, seq_len, hidden)
    # 用最后一步的输出算一个标量 loss（模拟 many-to-one 分类）
    loss = out[:, -1, :].sum()
    loss.backward()

    # 取第一个时间步的梯度范数
    return x.grad[:, 0, :].norm().item()

seq_lengths = [5, 10, 20, 50, 100, 200]
results = {rnn_type: [] for rnn_type in ['RNN', 'LSTM', 'GRU']}

for seq_len in seq_lengths:
    for rnn_type in ['RNN', 'LSTM', 'GRU']:
        grad_norm = compute_first_step_grad(rnn_type, seq_len)
        results[rnn_type].append(grad_norm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('梯度消失实验：不同 RNN 类型在不同序列长度下的第一步梯度范数', fontsize=11)

colors = {'RNN': 'C1', 'LSTM': 'C0', 'GRU': 'C2'}
for rnn_type, vals in results.items():
    axes[0].plot(seq_lengths, vals, 'o-', lw=2, color=colors[rnn_type], label=rnn_type)
    axes[1].plot(seq_lengths, vals, 'o-', lw=2, color=colors[rnn_type], label=rnn_type)

axes[0].set_title('线性坐标\n普通 RNN 梯度迅速趋近 0')
axes[0].set_xlabel('序列长度')
axes[0].set_ylabel('第一步梯度范数')
axes[0].legend()

axes[1].set_yscale('log')
axes[1].set_title('对数坐标\n更清楚地看出量级差距')
axes[1].set_xlabel('序列长度')
axes[1].set_ylabel('第一步梯度范数（对数刻度）')
axes[1].legend()

plt.tight_layout()
plt.show()

print("序列长度 → 第一步梯度范数对比：")
print(f"{'序列长度':>8}  {'RNN':>12}  {'LSTM':>12}  {'GRU':>12}")
for i, seq_len in enumerate(seq_lengths):
    print(f"{seq_len:>8}  {results['RNN'][i]:>12.2e}  {results['LSTM'][i]:>12.2e}  {results['GRU'][i]:>12.2e}")

print("\n观察：")
print("  普通 RNN：序列越长，第一步梯度越小，很快趋近 0")
print("  LSTM/GRU：梯度在长序列下仍保持一定量级，长程依赖得以学习")
print("  这就是 LSTM/GRU 能处理长序列而普通 RNN 不能的根本原因")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ── 演示 2：手动实现 LSTM Cell，追踪门控状态 ──────────────────
# 目标：直观看到遗忘门、输入门、输出门的值随时间步的变化

torch.manual_seed(0)

class LSTMCellManual(nn.Module):
    """手动实现的 LSTM Cell，便于追踪每个门的值"""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # 四个门（遗忘/输入/输出/候选状态）用一个大矩阵合并计算（效率更高）
        # 输出维度 = 4 * hidden_size，对应 [f, i, o, g] 四部分
        self.W = nn.Linear(input_size + hidden_size, 4 * hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x, h_prev, c_prev):
        # 拼接输入和上一步隐藏状态
        combined = torch.cat([h_prev, x], dim=1)  # (batch, hidden+input)
        gates = self.W(combined)                   # (batch, 4*hidden)

        H = self.hidden_size
        f_gate = torch.sigmoid(gates[:, :H])       # 遗忘门
        i_gate = torch.sigmoid(gates[:, H:2*H])    # 输入门
        o_gate = torch.sigmoid(gates[:, 2*H:3*H])  # 输出门
        g      = torch.tanh(gates[:, 3*H:])        # 候选细胞状态

        c_next = f_gate * c_prev + i_gate * g      # 更新细胞状态
        h_next = o_gate * torch.tanh(c_next)       # 更新隐藏状态

        return h_next, c_next, f_gate, i_gate, o_gate

# 构造一个简单的序列，故意在中间插入一个「重要事件」
# 序列：[背景噪声 × 5] + [重要事件] + [背景噪声 × 5]
input_size  = 8
hidden_size = 16
seq_len     = 12

x_seq = torch.randn(seq_len, 1, input_size) * 0.1  # 背景噪声（小幅度）
x_seq[5] = torch.randn(1, input_size) * 2.0         # 第 5 步：突然出现大幅度输入

lstm_cell = LSTMCellManual(input_size, hidden_size)

h = torch.zeros(1, hidden_size)
c = torch.zeros(1, hidden_size)

f_gates, i_gates, o_gates, c_norms, h_norms = [], [], [], [], []

for t in range(seq_len):
    h, c, f_g, i_g, o_g = lstm_cell(x_seq[t], h, c)
    f_gates.append(f_g.mean().item())
    i_gates.append(i_g.mean().item())
    o_gates.append(o_g.mean().item())
    c_norms.append(c.norm().item())
    h_norms.append(h.norm().item())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('LSTM 门控状态追踪\n（第 5 步出现大幅度输入，模拟"重要事件"）', fontsize=12)

steps = range(seq_len)

axes[0][0].bar(steps, [x_seq[t].norm().item() for t in range(seq_len)], color='gray')
axes[0][0].axvline(5, color='red', linestyle='--', lw=1.5, label='重要事件（第5步）')
axes[0][0].set_title('输入幅度（L2 范数）')
axes[0][0].set_xlabel('时间步')
axes[0][0].legend(fontsize=8)

axes[0][1].plot(steps, f_gates, 'o-', label='遗忘门 f', color='C1')
axes[0][1].plot(steps, i_gates, 's-', label='输入门 i', color='C0')
axes[0][1].plot(steps, o_gates, '^-', label='输出门 o', color='C2')
axes[0][1].axvline(5, color='red', linestyle='--', lw=1.5)
axes[0][1].set_title('三个门的平均激活值（0~1）\n接近 1 = 开，接近 0 = 关')
axes[0][1].set_xlabel('时间步')
axes[0][1].set_ylabel('门值（均值）')
axes[0][1].set_ylim(0, 1.1)
axes[0][1].legend(fontsize=8)

axes[1][0].plot(steps, c_norms, 'o-', color='C3', lw=2)
axes[1][0].axvline(5, color='red', linestyle='--', lw=1.5)
axes[1][0].set_title('细胞状态范数\n重要事件后细胞状态发生显著变化')
axes[1][0].set_xlabel('时间步')
axes[1][0].set_ylabel('C_t 的 L2 范数')

axes[1][1].plot(steps, h_norms, 's-', color='C4', lw=2)
axes[1][1].axvline(5, color='red', linestyle='--', lw=1.5)
axes[1][1].set_title('隐藏状态范数\n可以看到输出门过滤了细胞状态')
axes[1][1].set_xlabel('时间步')
axes[1][1].set_ylabel('h_t 的 L2 范数')

plt.tight_layout()
plt.show()

print("LSTM 门控行为观察（随机初始化，仅供直觉理解）：")
print("  重要事件（第5步）出现时，输入门激活程度变化 → 调整新信息写入量")
print("  细胞状态在第5步后发生变化，并在后续步骤中保持（长期记忆的体现）")
print("  输出门决定每步对外暴露多少细胞状态内容")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ── 演示 3：时间序列预测（RNN vs LSTM vs GRU 实战对比）─────────
# 任务：用前 T 个时间步预测第 T+1 步的值
# 数据：正弦波 + 多频率叠加 + 噪声，包含长程周期性

print("=== 时间序列预测：RNN vs LSTM vs GRU ===")

# ── 生成数据 ────────────────────────────────────────────────────
N = 2000
t = np.linspace(0, 8 * np.pi, N)
# 多频率叠加：快周期 + 慢周期 + 噪声
signal = (np.sin(t) + 0.5 * np.sin(3 * t) + 0.3 * np.sin(7 * t)
          + 0.1 * np.random.randn(N))
signal = (signal - signal.mean()) / signal.std()  # 标准化

SEQ_LEN = 40    # 用前 40 步预测下 1 步
TRAIN_RATIO = 0.8

# 构建滑动窗口数据集
X_all = np.array([signal[i:i+SEQ_LEN] for i in range(N - SEQ_LEN)])
y_all = signal[SEQ_LEN:]

split = int(len(X_all) * TRAIN_RATIO)
X_tr = torch.FloatTensor(X_all[:split]).unsqueeze(-1)   # (N_tr, seq, 1)
y_tr = torch.FloatTensor(y_all[:split])
X_te = torch.FloatTensor(X_all[split:]).unsqueeze(-1)
y_te = torch.FloatTensor(y_all[split:])

# ── 定义三种模型 ────────────────────────────────────────────────
class SeqModel(nn.Module):
    def __init__(self, rnn_type='LSTM', hidden=64, n_layers=2):
        super().__init__()
        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_size=1, hidden_size=hidden,
                           num_layers=n_layers, batch_first=True,
                           dropout=0.1 if n_layers > 1 else 0.0)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)         # out: (batch, seq, hidden)
        return self.head(out[:, -1, :]).squeeze()  # 取最后一步的隐藏状态

def train_model(rnn_type, epochs=50, batch=64, lr=1e-3):
    torch.manual_seed(0)
    model  = SeqModel(rnn_type=rnn_type)
    opt    = optim.Adam(model.parameters(), lr=lr)
    sched  = optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
    loss_fn = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_tr, y_tr)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=batch, shuffle=True)

    train_losses, test_losses = [], []
    for ep in range(epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
            opt.step()
            ep_loss += loss.item()
        sched.step()
        train_losses.append(ep_loss / len(loader))

        model.eval()
        with torch.no_grad():
            test_losses.append(loss_fn(model(X_te), y_te).item())

    return model, train_losses, test_losses

models_trained = {}
all_train_losses, all_test_losses = {}, {}

print("训练中（3 种模型各 50 轮）...")
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    m, tr_l, te_l = train_model(rnn_type)
    models_trained[rnn_type]   = m
    all_train_losses[rnn_type] = tr_l
    all_test_losses[rnn_type]  = te_l
    print(f"  {rnn_type:<6} 最终测试 MSE = {te_l[-1]:.4f}")

# ── 可视化 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('时间序列预测：RNN vs LSTM vs GRU', fontsize=13, fontweight='bold')

# 图1：训练曲线
colors = {'RNN': 'C1', 'LSTM': 'C0', 'GRU': 'C2'}
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    axes[0][0].plot(all_test_losses[rnn_type], lw=1.5,
                    color=colors[rnn_type], label=f'{rnn_type}')
axes[0][0].set_title('测试集 MSE 损失曲线')
axes[0][0].set_xlabel('Epoch')
axes[0][0].set_ylabel('MSE')
axes[0][0].legend()
axes[0][0].set_yscale('log')

# 图2~4：预测 vs 真实（测试集前 200 步）
view_len = 200
true_vals = y_te[:view_len].numpy()

for i, rnn_type in enumerate(['RNN', 'LSTM', 'GRU']):
    ax = axes[1][i] if i < 2 else axes[0][1]
    model = models_trained[rnn_type]
    model.eval()
    with torch.no_grad():
        pred = model(X_te[:view_len]).numpy()
    mse = np.mean((pred - true_vals) ** 2)
    ax.plot(true_vals, color='gray', lw=1,   label='真实值', alpha=0.8)
    ax.plot(pred,      color=colors[rnn_type], lw=1.5, label=f'{rnn_type} 预测')
    ax.set_title(f'{rnn_type} — 预测 vs 真实\nMSE = {mse:.4f}')
    ax.set_xlabel('时间步')
    ax.legend(fontsize=8)

# 隐藏多余子图
axes[1][2].axis('off')

plt.tight_layout()
plt.show()

print("\n=== 最终测试 MSE 对比 ===")
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    mse = all_test_losses[rnn_type][-1]
    print(f"  {rnn_type:<6}: {mse:.4f}")

print("\n观察：")
print("  普通 RNN 在长序列依赖上预测误差通常更大")
print("  LSTM 和 GRU 能捕捉多频率周期（慢周期需要记住很早的信息）")
print("  GRU 参数更少但性能接近 LSTM，是时间序列任务的常用选择")
print()
print("关于梯度裁剪（nn.utils.clip_grad_norm_）：")
print("  与梯度消失相对，RNN 也可能出现梯度爆炸（权重矩阵范数 > 1 时）")
print("  梯度裁剪将梯度范数限制在 max_norm 以内，防止参数更新步长过大")
print("  在 RNN 训练中是标准做法")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ── 演示 4：文本情感分类（简化版，不下载数据集）────────────────
# 用手工构造的情感词汇数据集演示 LSTM 文本分类的完整流程

print("=== 文本情感分类：LSTM Many-to-One ===")
print()

# 简单情感数据集（正面=1，负面=0）
sentences = [
    ("this movie is great and wonderful", 1),
    ("i love this amazing film very much", 1),
    ("fantastic performance by the cast", 1),
    ("excellent story and beautiful scenes", 1),
    ("highly recommend this masterpiece", 1),
    ("one of the best films ever made", 1),
    ("absolutely brilliant and moving", 1),
    ("a delightful and heartwarming story", 1),
    ("the acting was superb and impressive", 1),
    ("a truly wonderful cinematic experience", 1),
    ("this movie is terrible and boring", 0),
    ("i hate this awful waste of time", 0),
    ("worst film i have ever seen", 0),
    ("dull and disappointing throughout", 0),
    ("poor acting and bad screenplay", 0),
    ("completely unwatchable and painful", 0),
    ("nothing works in this disaster", 0),
    ("a terrible and forgettable film", 0),
    ("the story is weak and unconvincing", 0),
    ("an absolute mess with no redeeming qualities", 0),
]

# 构建词表
all_words = set()
for sent, _ in sentences:
    all_words.update(sent.split())

vocab = {word: idx + 2 for idx, word in enumerate(sorted(all_words))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1
vocab_size = len(vocab)

def encode(sentence, max_len=12):
    tokens = [vocab.get(w, 1) for w in sentence.split()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))  # padding
    return tokens

MAX_LEN = 12
X_data = torch.LongTensor([encode(s, MAX_LEN) for s, _ in sentences])
y_data = torch.FloatTensor([y for _, y in sentences])

# 80/20 划分
idx = torch.randperm(len(X_data))
split = int(0.8 * len(X_data))
X_tr, y_tr = X_data[idx[:split]], y_data[idx[:split]]
X_te, y_te = X_data[idx[split:]], y_data[idx[split:]]

# 定义情感分类 LSTM
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        emb = self.embedding(x)         # (batch, seq, embed_dim)
        out, (h_n, _) = self.lstm(emb)  # h_n: (1, batch, hidden)
        h = self.dropout(h_n.squeeze(0))
        return torch.sigmoid(self.head(h)).squeeze()

model = SentimentLSTM(vocab_size)
opt   = optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.BCELoss()

print("模型结构：")
print(f"  Embedding: vocab_size={vocab_size} → embed_dim=16")
print(f"  LSTM: 16 → hidden=32")
print(f"  Linear: 32 → 1 → Sigmoid")
print(f"  总参数量：{sum(p.numel() for p in model.parameters()):,}")
print()

train_losses, test_accs = [], []
EPOCHS = 100

for ep in range(EPOCHS):
    model.train()
    opt.zero_grad()
    loss = loss_fn(model(X_tr), y_tr)
    loss.backward()
    opt.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        preds = (model(X_te) > 0.5).float()
        acc = (preds == y_te).float().mean().item()
    test_accs.append(acc)

# 结果展示
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('情感分类 LSTM 训练过程', fontsize=12)
axes[0].plot(train_losses, color='C0', lw=1.5)
axes[0].set_title('训练 Loss (BCE)')
axes[0].set_xlabel('Epoch')
axes[1].plot(test_accs, color='C1', lw=1.5)
axes[1].set_title(f'测试准确率（最终：{test_accs[-1]:.2f}）')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

# 对新句子预测
print("=== 对新句子预测 ===")
test_sents = [
    "this is a wonderful and great story",
    "what a terrible and boring waste",
    "amazing film with brilliant acting",
    "poor and disappointing throughout",
]
model.eval()
with torch.no_grad():
    for sent in test_sents:
        x = torch.LongTensor([encode(sent, MAX_LEN)])
        prob = model(x).item()
        label = '正面 ✓' if prob > 0.5 else '负面 ✗'
        print(f"  [{label} {prob:.2f}] \"{sent}\"")

print()
print("完整流程说明：")
print("  1. Embedding 层：将每个词的 ID 映射为一个 embed_dim 维的向量（可学习）")
print("  2. LSTM 层：逐词处理序列，最后一步的隐藏状态 h_n 编码了整个句子的语义")
print("  3. Linear + Sigmoid：将句子向量映射为 0~1 的情感概率")
print("  这种 Many-to-One 的结构是 LSTM 用于文本分类的标准范式")